In [0]:
import dlt
from pyspark.sql.functions import col, coalesce, to_date, from_json, upper, expr, lit, current_timestamp

# -------------------------------------------------------------------
# Helper Functions
# -------------------------------------------------------------------
# Handles the 3 different date anomalies injected by the creators
def normalize_date(column_name):
    return coalesce(
        to_date(col(column_name), 'MM/dd/yyyy'),
        to_date(col(column_name), 'yyyy-MM-dd'),
        to_date(col(column_name), 'dd-MMM-yy')
    )

# -------------------------------------------------------------------
# Block 1: Master Dimensions & CDC (SCD Type 2)
# -------------------------------------------------------------------
# 1. Target table for CDC Customers (SCD Type 2)
dlt.create_streaming_table("silver_customers")

dlt.apply_changes(
    target = "silver_customers",
    source = "mavenmarket.bronze.customers", # Assuming your table name is customers
    keys = ["customer_id"],
    sequence_by = "_ingestion_timestamp", # Matches schema
    stored_as_scd_type = 2
)

# 2. Clean Stores
@dlt.table(name="silver_stores", comment="Cleaned store dimension")
def silver_stores():
    return (
        spark.readStream.table("mavenmarket.bronze.stores")
        .withColumn("store_type", upper(col("store_type")))
        .withColumn("first_opened_date", normalize_date("first_opened_date")) # string in schema
        # last_remodel_date is already a date type in schema
        .filter("total_sqft > 0 AND grocery_sqft <= total_sqft")
    )

# 3. Clean Promotions
@dlt.table(name="silver_promotions", comment="Cleaned promotions")
@dlt.expect_or_drop("valid_discount", "discount_percent >= 0 AND discount_percent <= 1.0")
@dlt.expect_or_drop("valid_dates", "end_date >= start_date")
@dlt.expect_or_drop("valid_id", "promotion_id IS NOT NULL")
def silver_promotions():
    return (
        spark.readStream.table("mavenmarket.bronze.promotions")
        .withColumn("start_date", normalize_date("start_date")) # string in schema
        # end_date is already a date type in schema
    )

# 4. Clean Regions
@dlt.table(name="silver_regions", comment="Cleaned regions dimension")
@dlt.expect_or_drop("valid_region_id", "region_id > 0 AND region_id < 999999")
@dlt.expect_or_drop("valid_district", "sales_district IS NOT NULL")
def silver_regions():
    return (
        spark.readStream.table("mavenmarket.bronze.regions")
        .withColumn("region_id", col("region_id").cast("int")) # Arrives as double
    )

# 5. Clean Products (Batch read because it is a full-load overwrite in Bronze)
@dlt.table(name="silver_products", comment="Cleaned products master data")
@dlt.expect_or_drop("valid_product_id", "product_id IS NOT NULL")
def silver_products():
    return (
        spark.read.table("mavenmarket.bronze.products")
        .withColumn("product_id", col("product_id").cast("int"))
        .withColumn("product_retail_price", col("product_retail_price").cast("double")) # Arrives as string
        .withColumn("product_weight", col("product_weight").cast("double")) # Arrives as string
    )

# 6. Clean Calendar
@dlt.table(name="silver_calendar", comment="Calendar dimension")
@dlt.expect_or_drop("valid_date", "calendar_date IS NOT NULL")
def silver_calendar():
    return spark.readStream.table("mavenmarket.bronze.calendar") # Matches file name


# -------------------------------------------------------------------
# Block 2: Unifying & Parsing Transactions (Kafka + ADLS)
# -------------------------------------------------------------------
# Define the JSON schema expected from Kafka orders
kafka_order_schema = "transaction_date STRING, stock_date STRING, product_id STRING, customer_id DOUBLE, store_id DOUBLE, transaction_mode STRING, quantity STRING"

@dlt.view(name="stg_kafka_orders")
def stg_kafka_orders():
    return (
        spark.readStream.table("mavenmarket.bronze.kafka_online_orders")
        .withColumn("payload", from_json(col("message_payload"), kafka_order_schema))
        .select(
            col("payload.transaction_date"),
            col("payload.stock_date"),
            col("payload.product_id"),
            col("payload.customer_id"),
            col("payload.store_id"),
            col("payload.transaction_mode"),
            col("payload.quantity")
        )
    )

@dlt.view(name="stg_adls_transactions")
def stg_adls_transactions():
    return spark.readStream.table("mavenmarket.bronze.transactions").select(
        "transaction_date", "stock_date", "product_id", "customer_id", "store_id", "transaction_mode", "quantity"
    )

@dlt.view(name="stg_all_transactions_unified")
def stg_all_transactions_unified():
    kafka_df = dlt.read_stream("stg_kafka_orders")
    adls_df = dlt.read_stream("stg_adls_transactions")
    
    return (
        kafka_df.unionByName(adls_df)
        .withColumn("transaction_date", normalize_date("transaction_date"))
        .withColumn("stock_date", normalize_date("stock_date"))
        .withColumn("quantity", col("quantity").cast("int"))
        .withColumn("customer_id", col("customer_id").cast("int")) # Cast from double
        .withColumn("store_id", col("store_id").cast("int")) # Cast from double
        .withColumn("product_id", col("product_id").cast("int")) # Cast from string
    )


# -------------------------------------------------------------------
# Block 3: Transaction Quarantine Pattern
# -------------------------------------------------------------------
@dlt.view(name="stg_transactions_with_dq_flags")
def stg_transactions_with_dq_flags():
    return (
        dlt.read_stream("stg_all_transactions_unified")
        .withColumn("is_valid_record", expr(
            "quantity > 0 AND "
            "product_id IS NOT NULL AND "
            "customer_id IS NOT NULL AND "
            "store_id IS NOT NULL AND "
            "transaction_date IS NOT NULL"
        ))
    )

@dlt.table(name="silver_transactions", comment="Clean unified transactions ready for Gold")
def silver_transactions():
    return (
        dlt.read_stream("stg_transactions_with_dq_flags")
        .filter("is_valid_record = TRUE")
        .drop("is_valid_record")
    )

@dlt.table(name="silver_transactions_quarantine", comment="Quarantined transactions with data quality failures")
def silver_transactions_quarantine():
    return (
        dlt.read_stream("stg_transactions_with_dq_flags")
        .filter("is_valid_record = FALSE")
        .drop("is_valid_record")
        .withColumn("quarantine_timestamp", current_timestamp())
    )


# -------------------------------------------------------------------
# Block 4: Returns and Inventory (Kafka)
# -------------------------------------------------------------------
@dlt.table(name="silver_returns", comment="Cleaned returns fact table")
@dlt.expect_or_drop("valid_quantity", "quantity > 0")
@dlt.expect_or_drop("valid_store_and_product", "store_id IS NOT NULL AND product_id IS NOT NULL")
def silver_returns():
    return (
        spark.readStream.table("mavenmarket.bronze.returns")
        .withColumn("return_date", normalize_date("return_date"))
        .withColumn("quantity", col("quantity").cast("int"))
        .withColumn("product_id", col("product_id").cast("int")) # Cast from string
        .withColumn("store_id", col("store_id").cast("int")) # Cast from double
    )

kafka_inventory_schema = "event_id STRING, restock_date STRING, product_id STRING, store_id STRING, restock_quantity STRING"

@dlt.table(name="silver_inventory", comment="Cleaned near-real-time inventory restocks")
@dlt.expect_or_drop("valid_restock", "restock_quantity > 0")
@dlt.expect_or_drop("valid_store", "store_id IS NOT NULL")
def silver_inventory():
    return (
        spark.readStream.table("mavenmarket.bronze.kafka_inventory")
        .withColumn("payload", from_json(col("message_payload"), kafka_inventory_schema))
        .select(
            col("payload.event_id"),
            normalize_date("payload.restock_date").alias("restock_date"),
            col("payload.product_id").cast("int").alias("product_id"),
            col("payload.store_id").cast("int").alias("store_id"),
            col("payload.restock_quantity").cast("int").alias("restock_quantity")
        )
    )